# 07 — Reward Shaping & Reward Hacking

The reward function is the specification of what you want the agent to do. Getting the specification right is as hard as — often harder than — choosing the right algorithm. Goodhart's Law in RL: 'When a measure becomes a target, it ceases to be a good measure.'

## Reward shaping

Reward shaping adds auxiliary rewards to guide the agent without changing the optimal policy. The key theoretical result (Ng et al., 1999): adding a **potential-based shaping reward** F(s,s') = γΦ(s') - Φ(s) to any environment reward does not change the optimal policy but can dramatically speed up learning.

**Examples of valid potential-based shaping**:
- Distance to goal (negative): Φ(s) = -d(s, goal)
- Intrinsic motivation / curiosity: Φ(s) = count-based novelty bonus

**Curiosity-driven exploration** (Pathak et al., 2017): add a reward bonus proportional to the prediction error of a forward model — the agent is rewarded for visiting states it cannot predict well. This works well in sparse-reward environments.

**Warning**: shaping rewards that are *not* potential-based do change the optimal policy and can lead to reward hacking.

## Reward hacking — Goodhart's Law in practice

Reward hacking is when the agent optimises the specified reward in a way that diverges from the intended goal.

**Classic examples**:
- Boat racing game: agent discovered it could score points by driving in circles collecting powerups without finishing the race
- CoastRunners: agent scored by catching powerups while on fire, never finishing
- Simulated robot: learned to make itself very tall to reach the goal sensor instead of walking

**RLHF reward hacking**: a reward model trained on human preferences can be fooled by outputs that score highly on the RM but are not actually preferred by humans. This is called **reward model overoptimisation**. The KL penalty in RLHF (NB 13) is the standard mitigation.

In [ ]:
# Reward design comparison and reward-hacker detector prototype
import numpy as np
import matplotlib.pyplot as plt

class TreasureGrid:
    def __init__(self, size=5, reward_mode="sparse"):
        self.size = size
        self.goal = (0, 4)
        self.traps = {(1,1), (2,3)}
        self.reward_mode = reward_mode
        self.reset()

    def state_id(self, pos): return pos[0] * self.size + pos[1]
    def reset(self):
        self.pos = (4, 0)
        return self.state_id(self.pos)

    def step(self, action):
        r, c = self.pos
        dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][action]
        nr = max(0, min(self.size-1, r+dr))
        nc = max(0, min(self.size-1, c+dc))
        self.pos = (nr, nc)
        s = self.state_id(self.pos)
        reached = self.pos == self.goal
        hit_trap = self.pos in self.traps
        done = reached or hit_trap
        reward = self._reward(reached, hit_trap)
        return s, reward, done

    def _reward(self, reached, hit_trap):
        if self.reward_mode == "sparse":
            return 1.0 if reached else (-1.0 if hit_trap else 0.0)
        elif self.reward_mode == "dense":
            dist = abs(self.pos[0] - self.goal[0]) + abs(self.pos[1] - self.goal[1])
            return 1.0 if reached else (-1.0 if hit_trap else -dist/8.0)
        elif self.reward_mode == "hacked":
            dist_center = abs(self.pos[0]-2) + abs(self.pos[1]-2)
            return 1.0 if reached else (0.5 if dist_center == 0 else -dist_center/8.0)

def q_learn(mode, n_episodes=3000):
    Q = np.zeros((25, 4))
    epsilon = 1.0
    success_rate = []
    successes = 0
    for ep in range(n_episodes):
        env = TreasureGrid(reward_mode=mode)
        s = env.reset()
        for _ in range(50):
            a = np.random.randint(4) if np.random.rand() < epsilon else np.argmax(Q[s])
            ns, r, done = env.step(a)
            Q[s,a] += 0.1 * (r + 0.95 * np.max(Q[ns]) - Q[s,a])
            s = ns
            if done:
                if env.pos == env.goal: successes += 1
                break
        epsilon = max(0.01, epsilon * 0.997)
        if (ep+1) % 100 == 0:
            success_rate.append(successes / 100)
            successes = 0
    return success_rate

results = {}
for mode in ["sparse", "dense", "hacked"]:
    results[mode] = q_learn(mode)
    print(f"{mode:8s}: final success rate = {np.mean(results[mode][-5:]):.1%}")

plt.figure(figsize=(10, 4))
for mode, color in [("sparse","gray"),("dense","steelblue"),("hacked","darkorange")]:
    plt.plot(results[mode], label=mode, color=color)
plt.xlabel("100-episode blocks"); plt.ylabel("Success rate")
plt.title("Effect of Reward Design on Learning", fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/reward_modes.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# reward-hacker: detect divergence between proxy and true reward

import numpy as np
import matplotlib.pyplot as plt

class RewardHackerDetector:
    def __init__(self, window=30):
        self.window = window
        self.proxy_rewards = []
        self.true_rewards = []

    def log(self, proxy, true): self.proxy_rewards.append(proxy); self.true_rewards.append(true)

    def divergence_score(self):
        if len(self.proxy_rewards) < 2 * self.window: return 0.0
        old_p = np.mean(self.proxy_rewards[-2*self.window:-self.window])
        new_p = np.mean(self.proxy_rewards[-self.window:])
        old_t = np.mean(self.true_rewards[-2*self.window:-self.window])
        new_t = np.mean(self.true_rewards[-self.window:])
        return max(0, (new_p - old_p) - (new_t - old_t))

    def plot(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(self.proxy_rewards, alpha=0.5, color='steelblue', label='Proxy reward')
        axes[0].plot(self.true_rewards, alpha=0.5, color='darkorange', label='True metric')
        axes[0].set_title('Proxy vs True Reward', fontweight='bold'); axes[0].legend()
        w = self.window
        corrs = [np.corrcoef(self.proxy_rewards[max(0,i-w):i], self.true_rewards[max(0,i-w):i])[0,1]
                 if i >= w else np.nan for i in range(len(self.proxy_rewards))]
        axes[1].plot(corrs, color='green')
        axes[1].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Danger zone')
        axes[1].set_title('Proxy-True Correlation (rolling)', fontweight='bold'); axes[1].legend()
        axes[1].set_ylim(-1.1, 1.1)
        plt.tight_layout(); plt.savefig('/tmp/reward_hacker.png', dpi=100, bbox_inches='tight'); plt.show()

np.random.seed(42)
detector = RewardHackerDetector(window=30)
for ep in range(200):
    if ep < 100:
        proxy = ep/100 * 0.8 + np.random.randn() * 0.1
        true = ep/100 * 0.75 + np.random.randn() * 0.1
    else:
        proxy = 0.8 + (ep-100)/100 * 0.4 + np.random.randn() * 0.05
        true = 0.75 + np.random.randn() * 0.1
    detector.log(proxy, true)

detector.plot()
div = detector.divergence_score()
print(f"Divergence score: {div:.3f} -- {'POSSIBLE HACKING' if div > 0.1 else 'OK'}")


## Practical guidance

1. Monitor proxy reward and true task metric separately throughout training
2. Use potential-based shaping when guiding learning — it is the only provably safe form
3. For RLHF: track KL divergence from the reference model; large KL signals reward model overoptimisation
4. Periodic human evaluation is the strongest mitigation — the proxy will drift from human intent eventually